# Dynamic Term Structure

**Purpose:** Introduce the dynamic term-structure helpers in `finstack_quant.core.market_data.dtsm` using a synthetic yield panel.

**Prerequisites:** Familiarity with yield curves, Nelson-Siegel factors, and the intuition behind PCA on curve changes.

**In this notebook:** We simulate a panel of curves, extract Diebold-Li factors, build a forecast, run PCA, and generate a simple shocked-curve scenario.


## Concept

This notebook keeps the term-structure workflow compact but complete:

1. Simulate a realistic panel of curves.
2. Recover the level, slope, and curvature factors.
3. Forecast the next curve from the factor dynamics.
4. Compare that factor view to a PCA shock view on yield changes.

### Convention: the decay parameter `lambda` is per YEAR

Every `dtsm` entry point takes **tenors in years** and a Nelson-Siegel decay
`lambda` expressed **per year**. The workspace default is **0.7308**.

Diebold & Li (2006) quote **0.0609**, but that is the **per-month** value from
their monthly Treasury panel. Feeding 0.0609 to year tenors is a silent unit
error, not a modelling choice: the curvature loading
\((1-e^{-\lambda\tau})/(\lambda\tau) - e^{-\lambda\tau}\) peaks at
\(\lambda\tau \approx 1.7935\), so the hump lands at
\(1.7935 / 0.0609 \approx 29.4\) years instead of
\(1.7935 / 0.7308 \approx 2.45\) years — the entire medium-tenor structure of
the curve disappears. Convert with \(0.0609 \times 12 = 0.7308\).

This is precisely the class of error that calling
`dtsm.nelson_siegel_yields` — rather than re-deriving the loadings in numpy —
exists to prevent: one parameterisation, documented once, in Rust.


In [1]:
import numpy as np

from finstack_quant.core.market_data import dtsm

# The Nelson-Siegel factor loadings live in Rust: `dtsm.nelson_siegel_yields`
# maps (lambda, (beta1, beta2, beta3), tenors) to fitted yields using the exact
# same loading matrix that `diebold_li_fit_factors` / `diebold_li_forecast`
# invert. Calling it here means the panel we simulate and the factors we recover
# below can never drift apart through a re-derived formula.


def simulate_panel(n_dates: int, tenors: np.ndarray, lam: float, rng: np.random.Generator) -> np.ndarray:
    """Simulate an AR(1) level/slope/curvature factor panel and map it to yields."""
    mu = np.array([0.040, -0.015, 0.005])
    phi = np.array([0.98, 0.95, 0.92])
    sigma = np.array([0.0015, 0.0020, 0.0025])
    betas = np.empty((n_dates, 3))
    betas[0] = mu
    for idx in range(1, n_dates):
        betas[idx] = mu + phi * (betas[idx - 1] - mu) + rng.normal(size=3) * sigma
    tenor_years = tenors.tolist()
    return np.vstack(
        [dtsm.nelson_siegel_yields(lam, (b[0], b[1], b[2]), tenor_years) for b in betas]
    )


## Factors, forecast, and scenario shock

The next cell works through the full script in one pass. It first builds a synthetic panel, then extracts factors and forecasts, and finally compares the factor approach with a PCA-based shock to the last observed curve.


In [2]:
rng = np.random.default_rng(seed=20260416)
tenors = np.array([0.25, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 20.0, 30.0])  # YEARS
lam = 0.7308  # per-YEAR decay (= 12 x Diebold-Li's per-MONTH 0.0609); hump at ~2.45y
yields = simulate_panel(200, tenors, lam, rng)

print(f"Yield panel shape  : {yields.shape}")
print(f"Mean 10Y yield     : {yields[:, 7].mean():.4%}")

factors = dtsm.diebold_li_fit_factors(tenors.tolist(), yields.tolist(), lam)
print("\nDiebold-Li factor summary")
print(f"beta1 mean/std     : {np.mean(factors['beta1']):+.4%} / {np.std(factors['beta1']):.4%}")
print(f"beta2 mean/std     : {np.mean(factors['beta2']):+.4%} / {np.std(factors['beta2']):.4%}")
print(f"beta3 mean/std     : {np.mean(factors['beta3']):+.4%} / {np.std(factors['beta3']):.4%}")
print(f"avg R^2            : {factors['r_squared_avg']:.6f}")

# Cross-check (pedagogical): the panel was generated from known betas via
# nelson_siegel_yields, so re-fitting must recover them to machine precision.
# R^2 ~= 1.0 confirms the forward map and the fit share one parameterisation.

forecast = dtsm.diebold_li_forecast(tenors.tolist(), yields.tolist(), 12, lam)
print("\n12-step forecast")
for tenor, last, predicted in zip(tenors, yields[-1], forecast['forecast_yields']):
    print(f"  {tenor:>5.2f}y last={last:.4%} forecast={predicted:.4%}")

changes = np.diff(yields, axis=0)
pca = dtsm.yield_pca_fit(changes.tolist(), 3)
print("\nPCA explained variance")
for idx, share in enumerate(pca['explained_variance_ratio'], start=1):
    print(f"  PC{idx}: {share:.4f}")
print(f"Cumulative through PC3: {pca['cumulative_variance'][-1]:.4f}")

shock = np.asarray(dtsm.yield_pca_scenario(changes.tolist(), 0, 2.0, 3))
shocked_curve = yields[-1] + shock
print("\nPC1 +2 sigma shock (bp)")
for tenor, delta in zip(tenors, shock):
    print(f"  {tenor:>5.2f}y delta={delta * 1e4:+.2f} bp")


Yield panel shape  : (200, 10)
Mean 10Y yield     : 3.2174%

Diebold-Li factor summary
beta1 mean/std     : +3.3336% / 0.5670%
beta2 mean/std     : -1.4245% / 0.5512%
beta3 mean/std     : +0.5771% / 0.6603%
avg R^2            : 1.000000

12-step forecast
   0.25y last=2.5257% forecast=2.0836%
   0.50y last=2.7139% forecast=2.2513%
   1.00y last=3.0034% forecast=2.5141%
   2.00y last=3.3497% forecast=2.8422%
   3.00y last=3.5223% forecast=3.0186%
   5.00y last=3.6551% forecast=3.1749%
   7.00y last=3.6932% forecast=3.2344%
  10.00y last=3.7104% forecast=3.2724%
  20.00y last=3.7237% forecast=3.3125%
  30.00y last=3.7278% forecast=3.3257%

PCA explained variance
  PC1: 0.9144
  PC2: 0.0742
  PC3: 0.0114
Cumulative through PC3: 1.0000

PC1 +2 sigma shock (bp)
   0.25y delta=+45.56 bp
   0.50y delta=+45.07 bp
   1.00y delta=+43.80 bp
   2.00y delta=+40.78 bp
   3.00y delta=+37.87 bp
   5.00y delta=+33.39 bp
   7.00y delta=+30.51 bp
  10.00y delta=+27.99 bp
  20.00y delta=+24.88 bp
  30.00y

## Takeaways

- The `dtsm` module supports both a factor-model view and a PCA view of the curve.
- Diebold-Li gives interpretable level, slope, and curvature factors, while PCA gives empirical shock directions.
- Forecasting and scenario generation are close neighbors in practice, so it is useful that both workflows live in the same API area.


In [3]:
{
    "avg_r_squared": round(factors['r_squared_avg'], 6),
    "pc3_cumulative": round(pca['cumulative_variance'][-1], 6),
    "final_10y": round(float(yields[-1][7]), 6),
    "shocked_10y": round(float(shocked_curve[7]), 6),
}


{'avg_r_squared': 1.0,
 'pc3_cumulative': 1.0,
 'final_10y': 0.037104,
 'shocked_10y': 0.039903}